# 04 Power Spectrum Pipeline

This notebook documents folded mesh construction, IA field generation, matter
density fields, and $P(k)$ measurement.


In [ ]:
import os
from pathlib import Path
import sys

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "ia_analysis").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the ia_analysis checkout.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RUNTIME_DIR = PROJECT_ROOT / ".notebook_runtime"
RUNTIME_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(RUNTIME_DIR / "matplotlib"))
os.environ.setdefault("IPYTHONDIR", str(RUNTIME_DIR / "ipython"))

print("Project root:", PROJECT_ROOT)


In [ ]:
from ia_analysis.spectra.ia_pk_cs import parse_pk_types, spec_keys_from_pk_types

pk_types = parse_pk_types("core")
spec_keys = spec_keys_from_pk_types(pk_types)

pk_types, spec_keys


In [ ]:
command = (
    "python -m ia_analysis.spectra.ia_pk_cs --flag GR --snap 21 "
    "--threads 8 --nmesh 512 --folds 1,2,4,8,16,32 "
    "--pk-types full --outdir outputs/pks"
)
print(command)


## Outputs

The main product is `pks_FLAG_SNAP.hdf5`.  Each sample contains folded spectra,
stitched native spectra, stitched target-$k$ spectra, and noise-corrected
spectra where available.


## Legacy Notebook Function Coverage

The functions below came from the former notebooks mapped to this workflow.
They remain visible here for compatibility and can be inspected without
executing the old notebooks' top-level data-loading cells.

### Pipeline and analysis functions

- `pks_pk_aia_nb.py`: `output_safe_name`, `format_z_title`, `compact_spectrum_label`, `compact_method_label`, `flag_sort_key`, `pks_path`, `parse_flag_snap_from_path`, `ensure_dir`, `apply_paper_grid`, `finite_median`, `infer_cosmo_box_metadata`, `omega_m_from_cosmo`, `growth_factor_fallback`, `growth_factor_D`, `ia_F_of_z`, `aia_prefactor`, `_get_group`, `_dataset_candidates`, `read_spectrum`, `read_spectrum_or_none`, `list_spectra`, `discover_samples_and_spectra`, `initialize_data_selection`, `pk_key_for_cov`, `available_cov_pk_types`, `covariance_group_has_spectrum`, `ensure_covariance`, `read_covariance_sigma`, `read_existing_sigma_or_covariance`, `sigma_for_spectrum`, `fallback_sigma`, `interpolate_logk`, `_local_window_bounds`, `moving_average_nan`, `smooth_curve`, `resample_smooth`, `prepare_signed_y`, `spectrum_math_label`, `pk_ylabel`, `sample_has_method`, `read_aia_estimator`, `run_pk_products`, `run_aia_products`
- `plot_all_pks_nb.py`: `snap3`, `pks_path`, `parse_flag_snap`, `discover_files`, `is_sample_group`, `list_samples`, `list_spectra`, `discover_samples_and_spectra`, `read_spectrum`, `split_spectrum_fields`, `pk_key_for_cov`, `spectrum_math_label`, `format_z_title`, `output_safe_name`, `prepare_y`, `_attr_float`, `infer_cosmo_box_metadata`, `available_cov_pk_types`, `covariance_group_has_spectrum`, `ensure_covariance`, `read_covariance_sigma`
- `plot_pks_nb.py`: `snap3`, `pks_path`, `parse_flag_snap`, `discover_files`, `is_sample_group`, `list_samples`, `list_spectra`, `discover_samples_and_spectra`, `read_spectrum`, `split_spectrum_fields`, `pk_key_for_cov`, `is_sample_independent_spectrum`, `display_sample_label`, `output_sample_label`, `spectrum_math_label`, `format_z_title`, `output_safe_name`, `prepare_y`, `resample_to_target_k`, `_attr_float`, `infer_cosmo_box_metadata`, `available_cov_pk_types`, `covariance_group_has_spectrum`, `_cov_kind_dataset_name`, `_extract_sigma_from_cov_payload`, `read_existing_covariance_payload`, `compute_covariance_payload_in_memory`, `ensure_covariance`, `read_covariance_sigma`, `omega_m_from_cosmo`, `growth_factor_fallback`, `growth_factor_D`, `aia_prefactor`, `read_spectrum_or_none`, `read_aia_estimator`, `aia_fit_value`, `sample_has_aia_method`, `choose_representative_sample_for_spectrum`

### Plotting functions

- `pks_pk_aia_nb.py`: `subplot_title`, `save_figure`, `make_model_legend`, `set_shared_labels`, `draw_series_with_band`, `load_plot_spectrum`, `plot_pk_model_grid`, `add_redshift_colorbar_to_axis`, `plot_pk_redshift_evolution_by_model`, `plot_all_pk`, `load_plot_aia`, `apply_aia_axis_style`, `plot_aia_model_grid`, `plot_aia_method_comparison`
- `plot_all_pks_nb.py`: `apply_pk_axis_style`, `save_figure`, `maybe_close`, `sigma_for_plot_grid`, `draw_pk_series`, `add_axis_colorbar`, `plot_model_comparison_grid`, `plot_redshift_evolution_by_model`, `plot_enhancement_relative_to_gr`, `plot_everything`
- `plot_pks_nb.py`: `apply_pk_axis_style`, `save_figure`, `maybe_close`, `sigma_for_plot_grid`, `draw_pk_series`, `add_axis_colorbar`, `plot_model_comparison_grid`, `plot_redshift_evolution_by_model`, `plot_enhancement_relative_to_gr`, `draw_aia_series`, `apply_aia_axis_style`, `plot_aia_model_comparison_grid`, `plot_aia_redshift_evolution_by_model`, `plot_aia_enhancement_relative_to_gr`, `plot_all_aia`, `plot_one_task`, `plot_everything`


In [ ]:
from IPython.display import Code, display
from ia_analysis.notebook_pipelines import legacy_catalog

LEGACY_EXPORTS = ('pks_pk_aia_nb.py', 'plot_all_pks_nb.py', 'plot_pks_nb.py')
legacy_manifest = legacy_catalog.manifest(LEGACY_EXPORTS)

print("Pipeline definitions:", len(legacy_manifest["pipeline"]))
print("Plotting definitions:", len(legacy_manifest["plotting"]))

def show_legacy_source(export, name, occurrence=1):
    """Display a preserved function or class from a former notebook."""
    text = legacy_catalog.source(export, name, occurrence=occurrence)
    display(Code(text, language="python"))
    return text

# Example:
# show_legacy_source(LEGACY_EXPORTS[0], legacy_manifest["pipeline"][0].name)
